### Notebook 2 — Nettoyage et transformation des données

In [34]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import to_date, col, round
from pyspark.sql.types import DoubleType, IntegerType
from pyspark.sql.functions import trim, initcap, upper, col

In [22]:
#Spark Session
spark = SparkSession.builder.appName("TradeCorp ETL").getOrCreate()

#### Afficher les csv et créer les dfs

In [23]:
DATA_PATH = "/home/jovyan/data"   # adapte selon ton volume Docker

# Lecture des fichiers CSV
df_customers      = spark.read.csv(f"{DATA_PATH}/customers.csv", header=True, inferSchema=True)
df_orders         = spark.read.csv(f"{DATA_PATH}/orders.csv", header=True, inferSchema=True)
df_order_details  = spark.read.csv(f"{DATA_PATH}/order_details.csv", header=True, inferSchema=True)
df_products       = spark.read.csv(f"{DATA_PATH}/products.csv", header=True, inferSchema=True)
df_categories     = spark.read.csv(f"{DATA_PATH}/categories.csv", header=True, inferSchema=True)
df_suppliers      = spark.read.csv(f"{DATA_PATH}/suppliers.csv", header=True, inferSchema=True)
df_employers      = spark.read.csv(f"{DATA_PATH}/employees.csv", header=True, inferSchema=True)
df_shippers       = spark.read.csv(f"{DATA_PATH}/shippers.csv", header=True, inferSchema=True)

dfs = {
    "orders": df_orders,
    "products": df_products,
    "customers": df_customers,
    "suppliers": df_suppliers,
    "categories": df_categories,
    "order_details": df_order_details,
    "employers": df_employers,
    "shippers": df_shippers
}

##### A11: Valeurs nulles

In [24]:
def count_nulls(df, name):
    results = []
    
    for c in df.columns:
        null_count = df.filter(col(c).isNull()).count()
        results.append((name, c, null_count))
    
    return spark.createDataFrame(results, ["dataframe", "column", "null_count"])

dfs = {
    "orders": df_orders,
    "products": df_products,
    "customers": df_customers,
    "suppliers": df_suppliers,
    "categories": df_categories,
    "order_details": df_order_details,
    "employers": df_employers,
    "shippers": df_shippers
}
summary_nulls = None

for name, df in dfs.items():
    df_nulls = count_nulls(df, name)
    summary_nulls = df_nulls if summary_nulls is None else summary_nulls.unionByName(df_nulls)
summary_nulls.show(truncate=False)

+---------+-----------------+----------+
|dataframe|column           |null_count|
+---------+-----------------+----------+
|orders   |order_id         |0         |
|orders   |customer_id      |0         |
|orders   |employee_id      |0         |
|orders   |order_date       |0         |
|orders   |required_date    |0         |
|orders   |shipped_date     |21        |
|orders   |ship_via         |0         |
|orders   |freight          |0         |
|orders   |ship_name        |0         |
|orders   |ship_address     |0         |
|orders   |ship_city        |0         |
|orders   |ship_region      |507       |
|orders   |ship_postal_code |19        |
|orders   |ship_country     |0         |
|products |product_id       |0         |
|products |product_name     |0         |
|products |supplier_id      |0         |
|products |category_id      |0         |
|products |quantity_per_unit|0         |
|products |unit_price       |0         |
+---------+-----------------+----------+
only showing top

##### A12 Supprimer les nulles et replacer par le médian

In [25]:
df_orders_clean = df_orders.filter(col("shipped_date").isNotNull())

In [27]:
# remplacer les valeurs nulles de unit_price par la médiane
median_price = df_products.approxQuantile("unit_price", [0.5], 0.01)[0]
print("Médiane de unit_price :", median_price)

#replacer

df_products_clean = df_products.fillna({"unit_price": median_price})
df_products_clean.count()

Médiane de unit_price : 19.45


77

##### A13 Cast des types

In [28]:
# df_orders → caster les dates en DateType
df_orders_casted = (
    df_orders
    .withColumn("order_date", to_date(col("order_date")))
    .withColumn("required_date", to_date(col("required_date")))
    .withColumn("shipped_date", to_date(col("shipped_date")))
)
#Vérification
df_orders_casted.printSchema()

root
 |-- order_id: integer (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- employee_id: integer (nullable = true)
 |-- order_date: date (nullable = true)
 |-- required_date: date (nullable = true)
 |-- shipped_date: date (nullable = true)
 |-- ship_via: integer (nullable = true)
 |-- freight: double (nullable = true)
 |-- ship_name: string (nullable = true)
 |-- ship_address: string (nullable = true)
 |-- ship_city: string (nullable = true)
 |-- ship_region: string (nullable = true)
 |-- ship_postal_code: string (nullable = true)
 |-- ship_country: string (nullable = true)



In [29]:
# df_order_details → caster les dates en DateType
df_order_details_casted = (
    df_order_details
    .withColumn("unit_price", col("unit_price").cast(DoubleType()))
    .withColumn("quantity", col("quantity").cast(IntegerType()))
)
df_order_details_casted.printSchema()

root
 |-- order_id: integer (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- discount: double (nullable = true)



##### A14 — Nettoyage des chaînes

In [30]:
# TRIM sur toutes les colonnes texte

df_customers_trimmed = df_customers

for c, t in df_customers.dtypes:
    if t == "string":
        df_customers_trimmed = df_customers_trimmed.withColumn(c, trim(col(c)))

# Mettre contact_name en Title Case avec initcap()
df_customers_trimmed = df_customers_trimmed.withColumn(
    "contact_name",
    initcap(col("contact_name"))
)
# Mettre country en MAJUSCULES avec upper()
df_customers_trimmed = df_customers_trimmed.withColumn(
    "country",
    upper(col("country"))
)

#Vérification
df_customers_trimmed.show(5, truncate=False)
df_customers_trimmed.printSchema()


+-----------+----------------------------------+------------------+--------------------+-----------------------------+-----------+------+-----------+-------+--------------+--------------+
|customer_id|company_name                      |contact_name      |contact_title       |address                      |city       |region|postal_code|country|phone         |fax           |
+-----------+----------------------------------+------------------+--------------------+-----------------------------+-----------+------+-----------+-------+--------------+--------------+
|ALFKI      |Alfreds Futterkiste               |Maria Anders      |Sales Representative|Obere Str. 57                |Berlin     |null  |12209      |GERMANY|030-0074321   |030-0076545   |
|ANATR      |Ana Trujillo Emparedados y helados|Ana Trujillo      |Owner               |Avda. de la Constitución 2222|México D.F.|null  |05021      |MEXICO |(5) 555-4729  |(5) 555-3745  |
|ANTON      |Antonio Moreno Taquería           |Antonio More

##### Q15 — Renommer les colonnes

In [33]:
# Renommer dans df_order_details
df_order_details_renamed = (
    df_order_details
        .withColumnRenamed("unit_price", "prix_unitaire")
        .withColumnRenamed("quantity", "quantite")
)

# Renommer dans df_orders
df_orders_renamed = (
    df_orders
        .withColumnRenamed("ship_via", "shipper_id")
)

# Vérification
df_order_details_renamed.printSchema()
df_orders_renamed.printSchema()


root
 |-- order_id: integer (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- prix_unitaire: double (nullable = true)
 |-- quantite: integer (nullable = true)
 |-- discount: double (nullable = true)

root
 |-- order_id: integer (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- employee_id: integer (nullable = true)
 |-- order_date: date (nullable = true)
 |-- required_date: date (nullable = true)
 |-- shipped_date: date (nullable = true)
 |-- shipper_id: integer (nullable = true)
 |-- freight: double (nullable = true)
 |-- ship_name: string (nullable = true)
 |-- ship_address: string (nullable = true)
 |-- ship_city: string (nullable = true)
 |-- ship_region: string (nullable = true)
 |-- ship_postal_code: string (nullable = true)
 |-- ship_country: string (nullable = true)



##### Q16 — Colonnes calculées

In [36]:
# 
df_order_details_final = (df_order_details_renamed
                          .withColumn("sous_total", round(col("prix_unitaire")
                                                          * col("quantite")
                                                          * (1 - col("discount")), 2)))
# Vérification
df_order_details_final.show(5, truncate=False)
df_order_details_final.printSchema()

+--------+----------+-------------+--------+--------+----------+
|order_id|product_id|prix_unitaire|quantite|discount|sous_total|
+--------+----------+-------------+--------+--------+----------+
|10248   |11        |14.0         |12      |0.0     |168.0     |
|10248   |42        |9.8          |10      |0.0     |98.0      |
|10248   |72        |34.8         |5       |0.0     |174.0     |
|10249   |14        |18.6         |9       |0.0     |167.4     |
|10249   |51        |42.4         |40      |0.0     |1696.0    |
+--------+----------+-------------+--------+--------+----------+
only showing top 5 rows

root
 |-- order_id: integer (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- prix_unitaire: double (nullable = true)
 |-- quantite: integer (nullable = true)
 |-- discount: double (nullable = true)
 |-- sous_total: double (nullable = true)



##### Q17 — Colonnes conditionnelles

In [39]:
#df_products → ajouter en_stock (True si units_in_stock > 0)

from pyspark.sql.functions import col

df_products_flag = (df_products.withColumn("en_stock", col("units_in_stock") > 0))

#df_orders → ajouter is_shipped (True si shipped_date n’est pas NULL)

df_orders_flag = (df_orders.withColumn("is_shipped", col("shipped_date").isNotNull()))

#Vérification

df_products_flag.show(3, truncate=False)
df_orders_flag.show(3, truncate=False)

+----------+-------------+-----------+-----------+-------------------+----------+--------------+--------------+-------------+------------+--------+
|product_id|product_name |supplier_id|category_id|quantity_per_unit  |unit_price|units_in_stock|units_on_order|reorder_level|discontinued|en_stock|
+----------+-------------+-----------+-----------+-------------------+----------+--------------+--------------+-------------+------------+--------+
|1         |Chai         |8          |1          |10 boxes x 30 bags |18.0      |39            |0             |10           |1           |true    |
|2         |Chang        |1          |1          |24 - 12 oz bottles |19.0      |17            |40            |25           |1           |true    |
|3         |Aniseed Syrup|1          |2          |12 - 550 ml bottles|10.0      |13            |70            |25           |0           |true    |
+----------+-------------+-----------+-----------+-------------------+----------+--------------+--------------+-

##### Q18 — Doublons

In [42]:
# 1. Vérifier le nombre total de lignes
total_rows = df_customers.count()
print("Nombre total de lignes :", total_rows)

# 2. Vérifier le nombre de lignes distinctes sur customer_id
distinct_rows = df_customers.select("customer_id").distinct().count()
print("Nombre de customer_id distincts :", distinct_rows)

# 3. Détecter s’il y a des doublons
if distinct_rows < total_rows:
    print(" Il existe des doublons sur customer_id.")
else:
    print(" Aucun doublon détecté sur customer_id.")
# 4. Supprimer les doublons si nécessaire

df_customers_clean = df_customers.dropDuplicates(["customer_id"])

# 5. Vérifications
print("Lignes après suppression des doublons :", df_customers_clean.count())


Nombre total de lignes : 91
Nombre de customer_id distincts : 91
 Aucun doublon détecté sur customer_id.
Lignes après suppression des doublons : 91


##### Q19 — Filtrage

##### Q20 — Sélection de colonnes